In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

# ===========================
# Potential function classes
# ===========================
class LJ:
    def __init__(self, epsilon=1.0, sigma=1.0):
        self.epsilon = epsilon
        self.sigma = sigma
    def __call__(self, r):
        return 4.0 * self.epsilon * ((self.sigma / r) ** 12 - (self.sigma / r) ** 6)


class PatchyLJ:
    def __init__(self, epsilon=1.0, sigma=1.0, omega=0.262, patches=None):
        self.epsilon = epsilon
        self.sigma = sigma
        self.omega = omega
        self.LJ_model = LJ(epsilon, sigma)
        self.patches = patches if patches is not None else [0, 1.5708, 3.14159, 4.71239]

    def wrap(self, ang):
        return (ang + np.pi) % (2 * np.pi) - np.pi

    def __call__(self, r, phi_i, phi_j):
        pot = 0
        for patch_i in self.patches:
            for patch_j in self.patches:
                theta_i = self.wrap(phi_i + patch_i)
                theta_j = self.wrap(phi_j + patch_j - np.pi)
                pot += self.LJ_model(r) * np.exp(-(theta_i**2 + theta_j**2) / (2 * self.omega**2))
        return pot


class OrientedLJ:
    def __init__(self, epsilon=1, sigma=1, m=1, n=2, alpha=np.pi):
        self.m = m
        self.n = n
        self.alpha = alpha
        self.LJ_model = LJ(epsilon, sigma)
        
    def __call__(self, r, phi1, phi2):
        radial = self.LJ_model(r)
        angular = 1 + self.m * np.cos(self.n * (phi2 - phi1) + self.alpha)
        return radial + angular


class MorseWithAngles:
    def __init__(self, De=1.0, re=8.5, a=0.5, C1=0.3, C2=0.1):
        self.De = De
        self.re = re
        self.a = a
        self.C1 = C1
        self.C2 = C2

    def __call__(self, r, phi1, phi2):
        radial = self.De * (np.exp(-2 * self.a * (r - self.re)) - 2 * np.exp(-self.a * (r - self.re)))
        angular = 1 + self.C1 * np.cos(np.radians(phi2 - phi1)) \
                    + self.C2 * np.cos(2 * np.radians(phi2 - phi1))
        return radial * angular


class GeometricLJ:
    def __init__(self, epsilon=1, sigma=1, pathcNum=3, rho=1, factor=0.5, S_h=1):
        self.patchNum = pathcNum
        self.rho = rho
        self.factor = factor
        self.S_h = S_h
        self.LJ_model = LJ(epsilon, sigma)
        
    def __call__(self, r, phi1, phi2):
        dx = r
        dy = 0
        total_potential = self.LJ_model(r)
        for i in range(self.patchNum):
            theta1 = phi1 + self.S_h* 2*np.pi*i/self.patchNum
            theta2 = phi2 + 2*np.pi*i/self.patchNum
            
            delta_cos = self.rho * (np.cos(theta2) - np.cos(theta1))
            delta_sin = self.rho * (np.sin(theta2) - np.sin(theta1))
            
            r_patch = np.sqrt((dx + delta_cos) * (dx + delta_cos) + (dy + delta_sin) * (dy + delta_sin))
            total_potential += self.factor * self.LJ_model(r_patch)
            
        return total_potential
    
    
# ===========================
#       Plotting helper
# ===========================

def plot_potential(potential, r_fixed=9.0, n_points=100):
    phi = np.linspace(0, 2*np.pi, n_points)
    phi1 = np.linspace(0, 2*np.pi, n_points)
    phi2 = np.linspace(0, 2*np.pi, n_points)
    phi1_grid, phi2_grid = np.meshgrid(phi1, phi2)
    
    r_vals = np.linspace(0.9, 3, n_points)

    # 1D plots
    pot_1d_phi = [potential(r_fixed, p, 0) for p in phi]
    pot_1d_r = [potential(r, 0, 0) for r in r_vals]
    fig, ax1 = plt.subplots(nrows=1, ncols=2, figsize=(10, 3))
    ax1[0].plot(phi, pot_1d_phi)
    ax1[0].set_xlabel("phi_1")
    ax1[0].set_ylabel("Potential")
    ax1[0].grid(True)
    
    ax1[1].plot(r_vals, pot_1d_r)
    ax1[1].set_xlabel("r")
    ax1[1].set_ylabel("Potential")
    ax1[1].grid(True)

    # 2D plot
    pot_2d = np.array([[potential(r_fixed, p1, p2) for p1, p2 in zip(row1, row2)]
                       for row1, row2 in zip(phi1_grid, phi2_grid)])
    fig, ax2 = plt.subplots(figsize=(6, 5))
    im = ax2.imshow(pot_2d, origin='lower', extent=[0, 2*np.pi, 0, 2*np.pi], aspect='auto')
    ax2.set_xlabel("phi_1")
    ax2.set_ylabel("phi_2")
    fig.colorbar(im, ax=ax2, label="Potential")

    plt.show()

plt.rcParams.update({
    'font.size': 9, 'axes.labelsize': 10,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'figure.dpi': 150,
    'savefig.dpi': 600, 'text.usetex': False, 'mathtext.default': 'regular'
})

In [ ]:

pot = MorseWithAngles(De=1.0, re=8.5, a=0.5, C1=0.3, C2=0.1)
# pot = OrientedLJ()
pot = PatchyLJ(epsilon=1.0, sigma=1.0, patches=[0, 2*np.pi/3, 4*np.pi/3])
# pot = GeometricLJ(pathcNum=4, S_h=1)

plot_potential(pot, r_fixed=7)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

interaction_type = 'SS'
data_path = f'/home/hadis/selfassembly/Data_Figures_Paper/E_all_{interaction_type}.dat'

CONFIG = {
    'energy_ref': -6227.1749,
    'zeta_max': 1.39687500,
    'phi2_max': 20
}

def process_data(filename):
    df = pd.read_csv(filename, sep=r'\s+', header=None,
                     names=['phi1', 'phi2', 'zeta', 'r', 'energy'])
    df['energy'] -= 2 * CONFIG['energy_ref']
    df['zeta'] = np.where(
        df['zeta'] < CONFIG['zeta_max'] / 2,
        df['zeta'],
        df['zeta'] - CONFIG['zeta_max']
    )
    return df

def get_min_energy_per_phi(df):
    return df.loc[df.groupby(['phi1', 'phi2'])['energy'].idxmin()].reset_index(drop=True)

SSdata_all = process_data(data_path)

In [ ]:
SSdata_all.head(5)
SSdata_all.tail(10)
SSdata_all.columns
SSdata_all.T
SSdata_all.sort_values(by='phi2', ascending=False)

for i in range(20):
    selected = SSdata_all[SSdata_all['phi1']==i]
     
selected_data = [SSdata_all[SSdata_all['phi1'] == i] for i in range(20)]

In [ ]:

# --- Process and extend data ---
SSdata_all = process_data(data_path)
SSdata = np.where(SSdata_all['r']==8)
# SSdata = screw_pbc(get_min_energy_per_phi(SSdata))
# SSdata_full = screw_pbc(SSdata, screw_direction=1)

# --- Pivot for plotting ---
grid_df = SSdata.pivot_table(index='phi2', columns='phi1', values='energy')
Phi1, Phi2 = np.meshgrid(grid_df.columns, grid_df.index)
E_grid = grid_df.values

# --- Plot ---
fig, ax = plt.subplots(figsize=(6,4))
c0 = ax.contourf(Phi1, Phi2, E_grid, levels=100, cmap='viridis')
ax.set_title("DFTB Energy Map")
ax.set_xlabel("phi1 (deg)")
ax.set_ylabel("phi2 (deg)")
fig.colorbar(c0, ax=ax, shrink=1)
plt.show()